# Loss Functions — Exercises

10 graded exercises covering the mathematics of loss functions for machine learning.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1–3 | Regression losses and MLE derivations |
| ★★ | 4–6 | Classification, KL divergence, calibration |
| ★★★ | 7–10 | Contrastive, RLHF, landscape, multi-task |

### Topic Map

| Topic | Exercise |
|---|---|
| MSE / MAE / Huber Bayes optimal | 1 |
| Gaussian NLL and MLE | 2 |
| Quantile regression | 3 |
| Cross-entropy and calibration | 4 |
| KL divergence: forward vs reverse | 5 |
| Focal loss and class imbalance | 6 |
| InfoNCE and temperature | 7 |
| DPO loss gradient analysis | 8 |
| Loss landscape and SAM | 9 |
| Multi-task uncertainty weighting | 10 |


In [ ]:
import numpy as np
import numpy.linalg as la
from scipy import stats

try:
    import matplotlib.pyplot as plt
    import matplotlib
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [10, 5]
    plt.rcParams['font.size'] = 12
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

COLORS = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#AA3377']

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got     : {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def huber(r, delta=1.0):
    """Huber loss on residual r."""
    return np.where(np.abs(r) <= delta,
                    0.5 * r**2,
                    delta * (np.abs(r) - 0.5 * delta))

def log_sum_exp(x):
    """Numerically stable log-sum-exp."""
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

print('Setup complete.')


---

## Exercise 1 ★ — Bayes-Optimal Predictions Under Different Losses

For a random variable $Y$ with distribution $P$, the **Bayes-optimal** constant prediction $\hat{y}^*$ minimises the expected loss:

$$\hat{y}^* = \arg\min_c \, \mathbb{E}[L(Y, c)]$$

**Tasks**

(a) Generate $n = 10{,}000$ samples from a **mixture of two Gaussians**:
$$Y \sim 0.3 \cdot \mathcal{N}(-3, 1) + 0.7 \cdot \mathcal{N}(2, 0.5^2)$$

(b) Verify numerically that the **mean** minimises MSE by sweeping $c \in [-5, 5]$.

(c) Verify that the **median** minimises MAE.

(d) Verify that the **0.9-quantile** minimises the quantile loss $\rho_{0.9}(r) = r(0.9 - \mathbf{1}[r < 0])$.

(e) Plot all three loss curves on one figure and mark the optima.


In [ ]:
# Exercise 1: Your Solution
np.random.seed(42)
n = 10_000

# (a) Sample from mixture
def sample_mixture(n):
    # YOUR CODE HERE
    pass

# (b) MSE sweep
def mse_sweep(samples, c_vals):
    # YOUR CODE HERE — return array of MSE values
    pass

# (c) MAE sweep
def mae_sweep(samples, c_vals):
    # YOUR CODE HERE
    pass

# (d) Quantile loss sweep (tau = 0.9)
def quantile_sweep(samples, c_vals, tau=0.9):
    # YOUR CODE HERE
    pass

samples = sample_mixture(n)
c_vals = np.linspace(-5, 5, 500)
mse_vals = mse_sweep(samples, c_vals)
mae_vals = mae_sweep(samples, c_vals)
q90_vals = quantile_sweep(samples, c_vals)
print('MSE minimiser:', None)
print('MAE minimiser:', None)
print('Q90 minimiser:', None)


In [ ]:
# Exercise 1: Solution
np.random.seed(42)
n = 10_000

def sample_mixture(n):
    mask = np.random.rand(n) < 0.3
    return np.where(mask,
                    np.random.normal(-3, 1, n),
                    np.random.normal(2, 0.5, n))

def mse_sweep(samples, c_vals):
    return np.array([np.mean((samples - c)**2) for c in c_vals])

def mae_sweep(samples, c_vals):
    return np.array([np.mean(np.abs(samples - c)) for c in c_vals])

def quantile_sweep(samples, c_vals, tau=0.9):
    losses = []
    for c in c_vals:
        r = samples - c
        losses.append(np.mean(r * (tau - (r < 0).astype(float))))
    return np.array(losses)

samples = sample_mixture(n)
c_vals = np.linspace(-5, 5, 500)
mse_vals = mse_sweep(samples, c_vals)
mae_vals = mae_sweep(samples, c_vals)
q90_vals = quantile_sweep(samples, c_vals)

mse_min = c_vals[np.argmin(mse_vals)]
mae_min = c_vals[np.argmin(mae_vals)]
q90_min = c_vals[np.argmin(q90_vals)]

header('Exercise 1: Bayes-Optimal Predictions')
print(f'Sample mean:     {samples.mean():.4f}')
print(f'MSE minimiser:   {mse_min:.4f}')
print(f'Sample median:   {np.median(samples):.4f}')
print(f'MAE minimiser:   {mae_min:.4f}')
print(f'Sample Q90:      {np.quantile(samples, 0.9):.4f}')
print(f'Q90 minimiser:   {q90_min:.4f}')

check_close('MSE minimiser ≈ mean', mse_min, samples.mean(), tol=0.05)
check_close('MAE minimiser ≈ median', mae_min, np.median(samples), tol=0.05)
check_close('Q90 minimiser ≈ 90th percentile', q90_min, np.quantile(samples, 0.9), tol=0.05)

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, vals, minimiser, title, color in zip(
        axes,
        [mse_vals, mae_vals, q90_vals],
        [mse_min, mae_min, q90_min],
        ['MSE', 'MAE', 'Quantile (τ=0.9)'],
        COLORS[:3]):
        ax.plot(c_vals, vals, color=color)
        ax.axvline(minimiser, color='k', ls='--', label=f'min={minimiser:.2f}')
        ax.set_title(title); ax.set_xlabel('c'); ax.legend()
    plt.suptitle('Loss curves vs constant prediction c', fontsize=13)
    plt.tight_layout(); plt.show()

print('\nTakeaway: Different losses select different statistics of P(Y) — '
      'MSE→mean, MAE→median, quantile loss→quantile. '
      'Loss choice encodes implicit assumptions about the data-generating process.')


---

## Exercise 2 ★ — Gaussian NLL and Heteroscedastic Regression

The **Gaussian negative log-likelihood** for a predicted mean $\mu$ and variance $\sigma^2$ is:

$$\mathcal{L}_{\text{NLL}} = \frac{(y - \mu)^2}{2\sigma^2} + \frac{1}{2}\log\sigma^2 + \text{const}$$

**Tasks**

(a) Implement `gaussian_nll(y, mu, log_sigma)` returning per-sample NLL.

(b) Show that minimising NLL over $\mu$ alone (fixed $\sigma$) recovers MSE.

(c) For **heteroscedastic** data generated as $y_i = x_i + \epsilon_i$ where $\epsilon_i \sim \mathcal{N}(0, (0.1 + 0.5|x_i|)^2)$, fit $(\mu(x_i), \sigma(x_i))$ by minimising total NLL over a grid of $\sigma$ values for each $x_i$.

(d) Compare the learned $\sigma(x)$ to the true standard deviation.


In [ ]:
# Exercise 2: Your Solution

def gaussian_nll(y, mu, log_sigma):
    """Per-sample Gaussian NLL. log_sigma = log(sigma)."""
    # YOUR CODE HERE
    pass

# (b) NLL at optimal mu should match MSE up to constant
y_test = np.array([1.0, 2.0, 3.0])
mu_test = np.array([1.5, 2.5, 3.5])
nll_fixed_sigma = None  # gaussian_nll with log_sigma=0
mse_manual = None
print('NLL (sigma=1):', nll_fixed_sigma)
print('0.5 * MSE    :', mse_manual)


In [ ]:
# Exercise 2: Solution

def gaussian_nll(y, mu, log_sigma):
    sigma2 = np.exp(2 * log_sigma)
    return 0.5 * ((y - mu)**2 / sigma2 + 2 * log_sigma + np.log(2 * np.pi))

# (b) sigma=1 → NLL = 0.5*(y-mu)^2 + const  (same as MSE)
y_test = np.array([1.0, 2.0, 3.0])
mu_test = np.array([1.5, 2.5, 3.5])
nll_vals = gaussian_nll(y_test, mu_test, log_sigma=0.0)
mse_vals2 = 0.5 * (y_test - mu_test)**2

header('Exercise 2: Gaussian NLL')
print('NLL (sigma=1):', nll_vals)
print('0.5 * MSE    :', mse_vals2 + 0.5 * np.log(2 * np.pi))
check_true('NLL equals 0.5*MSE + const when sigma=1',
           np.allclose(nll_vals - 0.5 * np.log(2 * np.pi), mse_vals2))

# (c) Heteroscedastic data
np.random.seed(42)
x = np.linspace(-3, 3, 200)
true_sigma = 0.1 + 0.5 * np.abs(x)
y = x + np.random.normal(0, true_sigma)

# Fit sigma per-point by grid search
log_sigma_grid = np.linspace(-3, 1, 200)
fitted_log_sigma = np.zeros(len(x))
for i in range(len(x)):
    nlls = gaussian_nll(y[i], x[i], log_sigma_grid)
    fitted_log_sigma[i] = log_sigma_grid[np.argmin(nlls)]
fitted_sigma = np.exp(fitted_log_sigma)

corr = np.corrcoef(fitted_sigma, true_sigma)[0, 1]
print(f'Correlation (fitted σ, true σ): {corr:.4f}')
check_true('Fitted sigma correlates with true sigma (r > 0.95)', corr > 0.95)

if HAS_MPL:
    plt.figure(figsize=(9, 4))
    plt.plot(x, true_sigma, color=COLORS[0], label='True σ(x)', lw=2)
    plt.plot(x, fitted_sigma, color=COLORS[1], ls='--', label='Fitted σ (NLL)', lw=2)
    plt.xlabel('x'); plt.ylabel('σ'); plt.title('Heteroscedastic NLL Fit')
    plt.legend(); plt.tight_layout(); plt.show()

print('\nTakeaway: Gaussian NLL enables heteroscedastic regression — '
      'the model learns input-dependent uncertainty, critical for calibrated predictions '
      'in LLM reward models and Bayesian neural networks.')


---

## Exercise 3 ★ — Quantile Regression and Prediction Intervals

The **quantile loss** (pinball loss) at level $\tau \in (0,1)$:

$$\rho_\tau(r) = r(\tau - \mathbf{1}[r < 0]),\quad r = y - \hat{y}$$

Minimising $\mathbb{E}[\rho_\tau(Y - c)]$ over constants $c$ gives the $\tau$-quantile of $P(Y)$.

**Tasks**

(a) Implement `pinball_loss(y, yhat, tau)` vectorised over samples.

(b) Generate 1000 samples from $Y \sim \text{Laplace}(0, 2)$ and find the empirical 10th, 50th, and 90th percentiles.

(c) Verify: minimising pinball loss at $\tau \in \{0.1, 0.5, 0.9\}$ recovers those percentiles.

(d) Construct a 80% **prediction interval** $[Q_{0.1}, Q_{0.9}]$ and verify empirical coverage.


In [ ]:
# Exercise 3: Your Solution

def pinball_loss(y, yhat, tau):
    """Mean pinball loss over samples."""
    # YOUR CODE HERE
    pass

np.random.seed(42)
y_lap = np.random.laplace(0, 2, 1000)

# Sweep c and find minimiser for each tau
c_grid = np.linspace(-10, 10, 1000)
tau_list = [0.1, 0.5, 0.9]
minimisers = {tau: None for tau in tau_list}  # fill in
print('Minimisers:', minimisers)


In [ ]:
# Exercise 3: Solution

def pinball_loss(y, yhat, tau):
    r = y - yhat
    return np.mean(r * (tau - (r < 0).astype(float)))

np.random.seed(42)
y_lap = np.random.laplace(0, 2, 1000)

c_grid = np.linspace(-10, 10, 1000)
tau_list = [0.1, 0.5, 0.9]

header('Exercise 3: Quantile Regression')
minimisers = {}
for tau in tau_list:
    losses = [pinball_loss(y_lap, c, tau) for c in c_grid]
    minimisers[tau] = c_grid[np.argmin(losses)]
    empirical = np.quantile(y_lap, tau)
    print(f'τ={tau}: minimiser={minimisers[tau]:.3f}, empirical Q={empirical:.3f}')
    check_close(f'pinball minimiser ≈ empirical Q{int(tau*100)}',
                minimisers[tau], empirical, tol=0.15)

# (d) Prediction interval coverage
q10, q90 = minimisers[0.1], minimisers[0.9]
coverage = np.mean((y_lap >= q10) & (y_lap <= q90))
print(f'\n80% PI [{q10:.2f}, {q90:.2f}], empirical coverage: {coverage:.3f}')
check_true('Empirical coverage is close to 80%', abs(coverage - 0.80) < 0.04)

print('\nTakeaway: Quantile loss enables distribution-free prediction intervals — '
      'used in conformal prediction and LLM uncertainty quantification to '
      'construct calibrated confidence sets without Gaussian assumptions.')


---

## Exercise 4 ★★ — Cross-Entropy, ECE, and Calibration

A model is **calibrated** if its predicted probabilities match empirical frequencies:

$$P(Y=1 \mid \hat{p} = p) = p \quad \forall p \in [0,1]$$

The **Expected Calibration Error** (ECE) measures deviation from calibration:

$$\text{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{n} \left| \text{acc}(B_b) - \text{conf}(B_b) \right|$$

**Tasks**

(a) Implement `binary_ce(y, p_hat)` — mean binary cross-entropy, numerically stable.

(b) Simulate a **miscalibrated** model: generate $y \sim \text{Bernoulli}(p_\text{true})$ but predict $\hat{p} = \text{sigmoid}(2 \cdot \text{logit}(p_\text{true}))$ (overconfident model).

(c) Implement `ece(y, p_hat, n_bins=10)` with equal-width bins $[0, 0.1), [0.1, 0.2), \ldots$

(d) Compare ECE of the overconfident model to a **perfectly calibrated** model.

(e) Plot a reliability diagram (confidence vs accuracy) for both models.


In [ ]:
# Exercise 4: Your Solution

def binary_ce(y, p_hat, eps=1e-7):
    """Numerically stable binary cross-entropy."""
    # YOUR CODE HERE
    pass

def ece(y, p_hat, n_bins=10):
    """Expected Calibration Error with equal-width bins."""
    # YOUR CODE HERE
    pass

np.random.seed(42)
n = 5000
p_true = np.random.uniform(0.05, 0.95, n)
y = (np.random.rand(n) < p_true).astype(float)

# Overconfident model
logit_true = np.log(p_true / (1 - p_true))
p_overconf = 1 / (1 + np.exp(-2 * logit_true))

print('ECE (overconfident):', ece(y, p_overconf))
print('ECE (calibrated)   :', ece(y, p_true))


In [ ]:
# Exercise 4: Solution

def binary_ce(y, p_hat, eps=1e-7):
    p = np.clip(p_hat, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def ece(y, p_hat, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    total_ece = 0.0
    n = len(y)
    for i in range(n_bins):
        mask = (p_hat >= bins[i]) & (p_hat < bins[i + 1])
        if mask.sum() == 0:
            continue
        acc = y[mask].mean()
        conf = p_hat[mask].mean()
        total_ece += (mask.sum() / n) * abs(acc - conf)
    return total_ece

np.random.seed(42)
n = 5000
p_true = np.random.uniform(0.05, 0.95, n)
y = (np.random.rand(n) < p_true).astype(float)

logit_true = np.log(p_true / (1 - p_true))
p_overconf = 1 / (1 + np.exp(-2 * logit_true))

ece_over = ece(y, p_overconf)
ece_cal  = ece(y, p_true)

header('Exercise 4: Calibration and ECE')
print(f'ECE (overconfident): {ece_over:.4f}')
print(f'ECE (calibrated)   : {ece_cal:.4f}')
check_true('Overconfident model has higher ECE', ece_over > ece_cal)
check_true('Calibrated model ECE < 0.02', ece_cal < 0.02)

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, p_hat, title in zip(axes,
                                [p_overconf, p_true],
                                [f'Overconfident (ECE={ece_over:.3f})',
                                 f'Calibrated (ECE={ece_cal:.3f})']):
        bins = np.linspace(0, 1, 11)
        bin_centers, accs, confs = [], [], []
        for i in range(10):
            mask = (p_hat >= bins[i]) & (p_hat < bins[i+1])
            if mask.sum() > 0:
                bin_centers.append((bins[i] + bins[i+1]) / 2)
                accs.append(y[mask].mean())
                confs.append(p_hat[mask].mean())
        ax.bar(bin_centers, accs, width=0.1, alpha=0.6, color=COLORS[0], label='Accuracy')
        ax.plot([0,1], [0,1], 'k--', label='Perfect calibration')
        ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
        ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()

print('\nTakeaway: ECE quantifies miscalibration — overconfident models '
      '(common in large NNs) have high ECE. Temperature scaling, '
      'a post-hoc linear rescaling of logits, is the standard fix in production LLMs.')


---

## Exercise 5 ★★ — KL Divergence: Forward vs Reverse

For distributions $P$ and $Q$:

$$D_{\text{KL}}(P \| Q) = \mathbb{E}_P\!\left[\log\frac{P}{Q}\right], \qquad D_{\text{KL}}(Q \| P) = \mathbb{E}_Q\!\left[\log\frac{Q}{P}\right]$$

These differ: **forward KL** (minimised by VI) is **mean-seeking** (covers all modes), while **reverse KL** is **mode-seeking** (collapses to one mode).

**Tasks**

(a) Implement `kl_discrete(p, q)` computing $D_{\text{KL}}(P \| Q)$ for discrete distributions.

(b) Verify non-negativity and asymmetry: compute both directions for two distinct distributions.

(c) Let the target $P$ be a **bimodal mixture** over $K=50$ discrete points, and $Q = \mathcal{N}(\mu, \sigma^2)$ discretised to the same grid. Find $\mu^*, \sigma^*$ minimising **forward KL** $D_{\text{KL}}(P \| Q)$ by grid search.

(d) Find $\mu^{**}, \sigma^{**}$ minimising **reverse KL** $D_{\text{KL}}(Q \| P)$.

(e) Show the two Gaussians differ — one is mean-seeking, one mode-seeking.


In [ ]:
# Exercise 5: Your Solution

def kl_discrete(p, q, eps=1e-12):
    """KL(P||Q) for discrete distributions. p,q are probability vectors."""
    # YOUR CODE HERE
    pass

# (b) Asymmetry
p_dist = np.array([0.1, 0.4, 0.5])
q_dist = np.array([0.3, 0.3, 0.4])
kl_pq = None
kl_qp = None
print(f'KL(P||Q) = {kl_pq}, KL(Q||P) = {kl_qp}')


In [ ]:
# Exercise 5: Solution

def kl_discrete(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p /= p.sum(); q /= q.sum()
    return np.sum(p * np.log(p / q))

# (b)
p_dist = np.array([0.1, 0.4, 0.5])
q_dist = np.array([0.3, 0.3, 0.4])
kl_pq = kl_discrete(p_dist, q_dist)
kl_qp = kl_discrete(q_dist, p_dist)

header('Exercise 5: KL Divergence')
print(f'KL(P||Q) = {kl_pq:.6f}')
print(f'KL(Q||P) = {kl_qp:.6f}')
check_true('KL >= 0 in both directions', kl_pq >= 0 and kl_qp >= 0)
check_true('KL is asymmetric', not np.isclose(kl_pq, kl_qp))

# (c-e) Bimodal target
x_grid = np.linspace(-6, 6, 100)
dx = x_grid[1] - x_grid[0]
p_bimodal = 0.5 * stats.norm.pdf(x_grid, -2, 0.8) + 0.5 * stats.norm.pdf(x_grid, 2, 0.8)
p_bimodal /= p_bimodal.sum()

mu_grid = np.linspace(-4, 4, 50)
sig_grid = np.linspace(0.3, 4.0, 50)

fwd_kl = np.zeros((len(mu_grid), len(sig_grid)))
rev_kl = np.zeros_like(fwd_kl)

for i, mu in enumerate(mu_grid):
    for j, sig in enumerate(sig_grid):
        q = stats.norm.pdf(x_grid, mu, sig)
        q /= q.sum()
        fwd_kl[i, j] = kl_discrete(p_bimodal, q)
        rev_kl[i, j] = kl_discrete(q, p_bimodal)

i_f, j_f = np.unravel_index(np.argmin(fwd_kl), fwd_kl.shape)
i_r, j_r = np.unravel_index(np.argmin(rev_kl), rev_kl.shape)
mu_fwd, sig_fwd = mu_grid[i_f], sig_grid[j_f]
mu_rev, sig_rev = mu_grid[i_r], sig_grid[j_r]

print(f'\nForward KL minimiser: μ={mu_fwd:.2f}, σ={sig_fwd:.2f} (mean-seeking → covers both modes)')
print(f'Reverse KL minimiser: μ={mu_rev:.2f}, σ={sig_rev:.2f} (mode-seeking → collapses to one mode)')
check_true('Forward KL sigma > reverse KL sigma (mean-seeking has larger spread)', sig_fwd > sig_rev)

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, mu_opt, sig_opt, title in zip(axes,
        [mu_fwd, mu_rev], [sig_fwd, sig_rev],
        [f'Forward KL (μ={mu_fwd:.1f}, σ={sig_fwd:.1f})',
         f'Reverse KL (μ={mu_rev:.1f}, σ={sig_rev:.1f})']):
        ax.plot(x_grid, p_bimodal, color=COLORS[0], label='Target P (bimodal)', lw=2)
        q_opt = stats.norm.pdf(x_grid, mu_opt, sig_opt)
        q_opt /= q_opt.sum()
        ax.plot(x_grid, q_opt, color=COLORS[1], ls='--', label='Approx Q (Gaussian)', lw=2)
        ax.set_title(title); ax.legend()
    plt.suptitle('Mode-covering vs Mode-seeking', fontsize=13)
    plt.tight_layout(); plt.show()

print('\nTakeaway: Variational inference uses reverse KL → mode-seeking (underestimates uncertainty). '
      'Forward KL → mean-seeking (over-spreads, used in distillation). '
      'Choice of KL direction has profound implications for RLHF and ELBO derivations.')


---

## Exercise 6 ★★ — Focal Loss and Class Imbalance

The **focal loss** modulates cross-entropy to down-weight easy examples:

$$\mathcal{L}_{\text{focal}} = -(1 - p_t)^\gamma \log p_t,
\quad p_t = \begin{cases} \hat{p} & y=1 \\ 1-\hat{p} & y=0 \end{cases}$$

**Tasks**

(a) Implement `focal_loss(y, p_hat, gamma)` returning mean focal loss.

(b) For $\gamma = 0$, verify that focal loss reduces to binary cross-entropy.

(c) Simulate imbalanced data: 950 negative, 50 positive samples. A naive classifier predicts $\hat{p} = 0.05$ for all. Compute total loss with CE vs focal ($\gamma=2$). Show that focal reduces the easy-negative contribution more.

(d) Plot the down-weighting factor $(1-p_t)^\gamma$ vs $p_t$ for $\gamma \in \{0.5, 1, 2, 5\}$.


In [ ]:
# Exercise 6: Your Solution

def focal_loss(y, p_hat, gamma=2.0, eps=1e-7):
    """Mean focal loss."""
    # YOUR CODE HERE
    pass

# (b) gamma=0 should match BCE
y_test = np.array([1.0, 0.0, 1.0, 0.0])
p_test = np.array([0.9, 0.1, 0.6, 0.4])
focal_0 = focal_loss(y_test, p_test, gamma=0)
bce_val = -np.mean(y_test * np.log(p_test + 1e-7) + (1-y_test) * np.log(1-p_test + 1e-7))
print(f'Focal(γ=0): {focal_0}, BCE: {bce_val}')


In [ ]:
# Exercise 6: Solution

def focal_loss(y, p_hat, gamma=2.0, eps=1e-7):
    p_hat = np.clip(p_hat, eps, 1 - eps)
    p_t = np.where(y == 1, p_hat, 1 - p_hat)
    return -np.mean((1 - p_t)**gamma * np.log(p_t))

y_test = np.array([1.0, 0.0, 1.0, 0.0])
p_test = np.array([0.9, 0.1, 0.6, 0.4])

header('Exercise 6: Focal Loss')
focal_0 = focal_loss(y_test, p_test, gamma=0)
bce_val = -np.mean(y_test * np.log(p_test) + (1-y_test) * np.log(1-p_test))
print(f'Focal(γ=0): {focal_0:.6f}')
print(f'BCE:        {bce_val:.6f}')
check_close('Focal(γ=0) == BCE', focal_0, bce_val)

# (c) Imbalanced
np.random.seed(42)
y_imb = np.array([1.0]*50 + [0.0]*950)
p_naive = np.full(1000, 0.05)

bce_total = focal_loss(y_imb, p_naive, gamma=0)
focal_total = focal_loss(y_imb, p_naive, gamma=2)

# Separate easy negatives (pt ≈ 0.95, weight = (1-0.95)^2 = 0.0025)
pos_mask = y_imb == 1
neg_mask = ~pos_mask
bce_neg   = focal_loss(y_imb[neg_mask], p_naive[neg_mask], gamma=0)
focal_neg = focal_loss(y_imb[neg_mask], p_naive[neg_mask], gamma=2)

print(f'\nImbalanced (950 neg / 50 pos), ŷ=0.05:')
print(f'  BCE total:       {bce_total:.4f}   |  BCE neg contribution:   {bce_neg:.4f}')
print(f'  Focal total:     {focal_total:.4f}   |  Focal neg contribution: {focal_neg:.4f}')
print(f'  Neg suppression ratio: {bce_neg / focal_neg:.1f}x')
check_true('Focal suppresses easy negatives more than BCE', focal_neg < bce_neg)

if HAS_MPL:
    pt_vals = np.linspace(0.01, 0.99, 300)
    plt.figure(figsize=(8, 5))
    for gamma, color in zip([0.5, 1.0, 2.0, 5.0], ['#0077BB','#EE7733','#009988','#CC3311']):
        plt.plot(pt_vals, (1 - pt_vals)**gamma, color=color, label=f'γ={gamma}')
    plt.xlabel('$p_t$'); plt.ylabel('$(1-p_t)^\\gamma$')
    plt.title('Focal loss down-weighting factor'); plt.legend(); plt.tight_layout(); plt.show()

print('\nTakeaway: Focal loss (Lin et al. 2017, RetinaNet) down-weights easy examples by '
      '(1-p_t)^γ. With γ=2, a well-classified negative (p_t=0.9) contributes '
      '100× less loss than an ambiguous one (p_t=0.5), enabling training on '
      'heavily imbalanced datasets without hard negative mining.')


---

## Exercise 7 ★★★ — InfoNCE and Temperature Sensitivity

The **InfoNCE loss** (used in contrastive learning, CLIP) is:

$$\mathcal{L}_{\text{NCE}} = -\frac{1}{N}\sum_{i=1}^N \log \frac{\exp(s_{ii}/\tau)}{\sum_{j=1}^N \exp(s_{ij}/\tau)}$$

where $s_{ij} = \mathbf{z}_i^\top \mathbf{z}_j'$ are similarity scores and $\tau > 0$ is temperature.

**Tasks**

(a) Implement `infonce_loss(S, tau)` where $S \in \mathbb{R}^{N \times N}$ is a similarity matrix. Use numerically stable log-softmax.

(b) Verify: when $S = I$ (identity, perfect alignment), loss equals $\log N$.

(c) Generate random unit-vector embeddings ($N=32$, $d=64$). Add 20% corrupted positives (randomly flipped). Show how loss and gradient magnitude change with $\tau \in \{0.01, 0.1, 0.5, 1.0, 2.0\}$.

(d) Verify the gradient of InfoNCE equals the difference between the positive similarity and the expected similarity under the softmax distribution.


In [ ]:
# Exercise 7: Your Solution

def infonce_loss(S, tau):
    """InfoNCE loss. S is NxN similarity matrix. Numerically stable."""
    # YOUR CODE HERE
    pass

# (b) Identity similarity
N = 8
S_identity = np.eye(N)
loss_identity = infonce_loss(S_identity, tau=1.0)
print(f'Loss on identity S: {loss_identity:.4f}, expected log(N) = {np.log(N):.4f}')


In [ ]:
# Exercise 7: Solution

def log_softmax(x):
    m = x.max(axis=-1, keepdims=True)
    return x - m - np.log(np.sum(np.exp(x - m), axis=-1, keepdims=True))

def infonce_loss(S, tau):
    N = S.shape[0]
    log_p = log_softmax(S / tau)
    return -np.mean(log_p[np.arange(N), np.arange(N)])

header('Exercise 7: InfoNCE')
N = 8
S_identity = np.eye(N)
loss_id = infonce_loss(S_identity, tau=1.0)
print(f'Loss on identity S (τ=1): {loss_id:.6f}')
print(f'log(N) = {np.log(N):.6f}')
check_close('InfoNCE on identity == log(N)', loss_id, np.log(N))

# (c) Random embeddings with corrupted positives
np.random.seed(42)
N_c, d = 32, 64
Z = np.random.randn(N_c, d)
Z /= la.norm(Z, axis=1, keepdims=True)  # unit vectors
Zp = Z.copy()
corrupt_idx = np.random.choice(N_c, size=int(0.2 * N_c), replace=False)
Zp[corrupt_idx] = np.random.randn(len(corrupt_idx), d)
Zp /= la.norm(Zp, axis=1, keepdims=True)

S = Z @ Zp.T

print('\nTemperature sweep:')
taus = [0.01, 0.1, 0.5, 1.0, 2.0]
losses = [infonce_loss(S, tau) for tau in taus]
for tau, loss in zip(taus, losses):
    print(f'  τ={tau:4.2f}  loss={loss:.4f}')

# (d) Gradient verification
tau = 0.1
softmax_row = lambda si: np.exp(si/tau) / np.exp(si/tau).sum()
# Analytic: dL/ds_ii = (softmax_ii - 1) / (N*tau)
eps_val = 1e-5
analytic_grad = []
numerical_grad = []
for i in range(min(5, N_c)):
    s_row = S[i]
    a_grad = (softmax_row(s_row)[i] - 1) / (N_c * tau)
    S_plus = S.copy(); S_plus[i, i] += eps_val
    S_minus = S.copy(); S_minus[i, i] -= eps_val
    n_grad = (infonce_loss(S_plus, tau) - infonce_loss(S_minus, tau)) / (2 * eps_val)
    analytic_grad.append(a_grad)
    numerical_grad.append(n_grad)

grad_err = np.max(np.abs(np.array(analytic_grad) - np.array(numerical_grad)))
print(f'\nMax gradient error (analytic vs numerical): {grad_err:.2e}')
check_true('Gradient error < 1e-7', grad_err < 1e-7)

print('\nTakeaway: Low temperature (τ→0) sharpens the softmax → harder negatives, '
      'higher loss, steeper gradients. CLIP uses τ≈0.07 (learned), '
      'while SimCLR uses τ=0.5. Temperature is a critical hyperparameter '
      'controlling uniformity vs alignment trade-off.')


---

## Exercise 8 ★★★ — DPO Loss and Bradley-Terry Preference Model

**Direct Preference Optimisation** (Rafailov et al. 2023) derives a loss from the Bradley-Terry preference model. For preferred response $y_w$ and rejected $y_l$:

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\!\left(\beta \left[\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)$$

**Tasks**

(a) Implement `dpo_loss(log_ratio_w, log_ratio_l, beta)` where `log_ratio_k` $= \log\pi_\theta(y_k|x) - \log\pi_{\text{ref}}(y_k|x)$.

(b) Show that the gradient w.r.t. `log_ratio_w` is always negative (increasing preferred likelihood decreases loss) and w.r.t. `log_ratio_l` is always positive.

(c) Simulate a batch of 100 preference pairs. Start with zero log-ratios (model = reference). Perform 200 gradient steps (analytic gradient) and show the margin $\log\pi_\theta(y_w)/\pi_{\text{ref}}(y_w) - \log\pi_\theta(y_l)/\pi_{\text{ref}}(y_l)$ increasing over training.

(d) Show that DPO implicitly acts as a reward model: the implicit reward is $r_\theta(x,y) = \beta\log\pi_\theta(y|x)/\pi_{\text{ref}}(y|x)$.


In [ ]:
# Exercise 8: Your Solution

def dpo_loss(log_ratio_w, log_ratio_l, beta=0.1):
    """DPO loss. Inputs are 1D arrays of per-pair log ratios."""
    # YOUR CODE HERE
    pass

# (b) Gradient check
lr_w = np.array([0.5])
lr_l = np.array([-0.5])
# YOUR CODE HERE: compute gradient of dpo_loss w.r.t. lr_w numerically
grad_w = None
print(f'Gradient w.r.t log_ratio_w: {grad_w} (should be negative)')


In [ ]:
# Exercise 8: Solution

def sigmoid(x): return 1 / (1 + np.exp(-x))

def dpo_loss(log_ratio_w, log_ratio_l, beta=0.1):
    margin = beta * (log_ratio_w - log_ratio_l)
    return -np.mean(np.log(sigmoid(margin)))

header('Exercise 8: DPO Loss')

# (b) Gradient signs
eps_val = 1e-5
lr_w = np.array([0.5]); lr_l = np.array([-0.5])
grad_w = (dpo_loss(lr_w + eps_val, lr_l) - dpo_loss(lr_w - eps_val, lr_l)) / (2*eps_val)
grad_l = (dpo_loss(lr_w, lr_l + eps_val) - dpo_loss(lr_w, lr_l - eps_val)) / (2*eps_val)
print(f'∂L/∂log_ratio_w = {grad_w:.6f} (should be < 0)')
print(f'∂L/∂log_ratio_l = {grad_l:.6f} (should be > 0)')
check_true('Gradient w.r.t. log_ratio_w < 0', grad_w < 0)
check_true('Gradient w.r.t. log_ratio_l > 0', grad_l > 0)

# (c) Simulate training
np.random.seed(42)
n_pairs = 100
beta = 0.1
lr = 0.05

# log-ratios start at 0 (model = reference)
lr_w_train = np.zeros(n_pairs)
lr_l_train = np.zeros(n_pairs)

margins_hist = []
losses_hist = []

for step in range(200):
    margin = beta * (lr_w_train - lr_l_train)
    sig_neg = sigmoid(-margin)
    # analytic gradient: dL/d(margin) = -mean(sig_neg), chain rule to lr_w and lr_l
    grad_margin = -sig_neg / n_pairs  # per-sample
    lr_w_train -= lr * beta * grad_margin
    lr_l_train += lr * beta * grad_margin
    margins_hist.append((lr_w_train - lr_l_train).mean())
    losses_hist.append(dpo_loss(lr_w_train, lr_l_train, beta))

print(f'\nFinal margin: {margins_hist[-1]:.4f} (started at 0.0)')
print(f'Final loss:   {losses_hist[-1]:.4f}')
check_true('Margin increases during training', margins_hist[-1] > margins_hist[0])
check_true('Loss decreases during training', losses_hist[-1] < losses_hist[0])

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(margins_hist, color=COLORS[0])
    ax1.set_title('Preference margin over training'); ax1.set_xlabel('Step')
    ax2.plot(losses_hist, color=COLORS[1])
    ax2.set_title('DPO loss over training'); ax2.set_xlabel('Step')
    plt.tight_layout(); plt.show()

# (d) Implicit reward
implicit_reward_w = beta * lr_w_train.mean()
implicit_reward_l = beta * lr_l_train.mean()
print(f'\nImplicit reward r(y_w): {implicit_reward_w:.4f} (should be > r(y_l))')
print(f'Implicit reward r(y_l): {implicit_reward_l:.4f}')
check_true('Preferred implicit reward > rejected', implicit_reward_w > implicit_reward_l)

print('\nTakeaway: DPO eliminates the explicit reward model by reparameterising '
      'the RLHF objective. The gradient is proportional to -σ(-β·margin), '
      'up-weighting hard preference pairs. Used in LLaMA-3, Mistral, Phi-3.')


---

## Exercise 9 ★★★ — Loss Landscape and Sharpness-Aware Minimisation

The **sharpness** of a minimum measures how much the loss increases in the worst-case neighbourhood:

$$\text{Sharpness}(\theta, \rho) = \max_{\|\epsilon\|_2 \leq \rho} \mathcal{L}(\theta + \epsilon) - \mathcal{L}(\theta)$$

**SAM** finds a flat minimum by solving:
$$\hat{\epsilon}(\theta) = \rho \cdot \frac{\nabla_\theta \mathcal{L}}{\|\nabla_\theta \mathcal{L}\|},
\qquad \theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}(\theta + \hat{\epsilon})$$

**Tasks**

(a) Define a 2D quadratic landscape $\mathcal{L}(w_1, w_2) = w_1^2 + 100 w_2^2$ (ill-conditioned Hessian). Implement `sharpness(w, loss_fn, rho, n_samples=500)` by random sampling in the $\ell_2$ ball.

(b) Compare sharpness at two minima: $\mathbf{w}_A = (0, 0)$ (true minimum) and $\mathbf{w}_B = (0.1, 0.01)$ (nearby point). Which has higher sharpness?

(c) Implement one SAM step from starting point $(1, 0.1)$ with $\rho = 0.3$. Verify the perturbation vector $\hat{\epsilon}$ is in the gradient direction.

(d) Run 100 SGD steps and 100 SAM steps from the same start. Compare the final sharpness values.


In [ ]:
# Exercise 9: Your Solution

def loss_fn(w):
    """2D loss: w[0]^2 + 100*w[1]^2"""
    return w[0]**2 + 100 * w[1]**2

def grad_fn(w):
    """Gradient of loss_fn."""
    return np.array([2*w[0], 200*w[1]])

def sharpness(w, loss_fn, rho, n_samples=500):
    """Approximate sharpness via random sampling in l2 ball."""
    # YOUR CODE HERE
    pass

w_A = np.array([0.0, 0.0])
w_B = np.array([0.1, 0.01])
rho = 0.3
print(f'Sharpness at A: {sharpness(w_A, loss_fn, rho)}')
print(f'Sharpness at B: {sharpness(w_B, loss_fn, rho)}')


In [ ]:
# Exercise 9: Solution

def loss_fn(w):
    return w[0]**2 + 100 * w[1]**2

def grad_fn(w):
    return np.array([2*w[0], 200*w[1]])

def sharpness(w, loss_fn, rho, n_samples=500):
    np.random.seed(0)
    eps_samples = np.random.randn(n_samples, len(w))
    eps_samples /= la.norm(eps_samples, axis=1, keepdims=True)
    eps_samples *= rho
    perturbed_losses = np.array([loss_fn(w + eps) for eps in eps_samples])
    return np.max(perturbed_losses) - loss_fn(w)

header('Exercise 9: Loss Landscape and SAM')
w_A = np.array([0.0, 0.0])
w_B = np.array([0.1, 0.01])
rho = 0.3

sharp_A = sharpness(w_A, loss_fn, rho)
sharp_B = sharpness(w_B, loss_fn, rho)
print(f'Sharpness at A (true min): {sharp_A:.4f}')
print(f'Sharpness at B (off-min):  {sharp_B:.4f}')
check_true('A has lower sharpness than B', sharp_A < sharp_B)

# (c) SAM step
w_start = np.array([1.0, 0.1])
g = grad_fn(w_start)
eps_hat = rho * g / la.norm(g)
print(f'\nSAM perturbation: {eps_hat}')
# eps_hat should be in gradient direction
cos_sim = np.dot(eps_hat, g) / (la.norm(eps_hat) * la.norm(g))
check_close('Perturbation is in gradient direction (cos=1)', cos_sim, 1.0)

# (d) SGD vs SAM training
np.random.seed(42)
eta = 0.01
n_steps = 100

# SGD
w_sgd = np.array([1.0, 0.1])
for _ in range(n_steps):
    w_sgd -= eta * grad_fn(w_sgd)

# SAM
w_sam = np.array([1.0, 0.1])
for _ in range(n_steps):
    g = grad_fn(w_sam)
    eps = rho * g / (la.norm(g) + 1e-12)
    g_perturbed = grad_fn(w_sam + eps)
    w_sam -= eta * g_perturbed

sharp_sgd = sharpness(w_sgd, loss_fn, rho)
sharp_sam = sharpness(w_sam, loss_fn, rho)
print(f'\nAfter {n_steps} steps:')
print(f'  SGD final loss: {loss_fn(w_sgd):.6f}, sharpness: {sharp_sgd:.4f}')
print(f'  SAM final loss: {loss_fn(w_sam):.6f}, sharpness: {sharp_sam:.4f}')
check_true('SAM sharpness <= SGD sharpness', sharp_sam <= sharp_sgd)

print('\nTakeaway: SAM finds flatter minima with better generalisation. '
      'Hochreiter & Schmidhuber (1997) showed flat minima generalise better; '
      'SAM (Foret et al. 2021) makes this tractable. Used in PaLM, ViT training.')


---

## Exercise 10 ★★★ — Multi-Task Uncertainty Weighting

Kendall et al. (2018) derive a principled multi-task weighting from homoscedastic uncertainty. For two tasks with Gaussian likelihoods:

$$\mathcal{L}_{\text{multi}} = \frac{1}{2\sigma_1^2}\mathcal{L}_1 + \frac{1}{2\sigma_2^2}\mathcal{L}_2 + \log\sigma_1 + \log\sigma_2$$

The optimal $\sigma_k^*$ satisfies $\sigma_k^{*2} = \mathcal{L}_k / \text{(normalising const)}$, making high-loss tasks automatically down-weighted.

**Tasks**

(a) Implement `multi_task_loss(L1, L2, log_sigma1, log_sigma2)` computing the formula above.

(b) Verify the gradient w.r.t. $\log\sigma_1$ is zero at the optimal value $\sigma_1^* = \sqrt{\mathcal{L}_1 / 2}$. (Gradient: $-\mathcal{L}_1/\sigma_1^2 + 1 = 0 \Rightarrow \sigma_1^{*2} = \mathcal{L}_1$.)

(c) Simulate a training scenario with two tasks: Task 1 loss $\mathcal{L}_1 = 5.0$, Task 2 loss $\mathcal{L}_2 = 0.5$. Run 200 steps optimising only $\sigma_1, \sigma_2$ (keeping $\mathcal{L}_k$ fixed). Show that $\sigma_1 > \sigma_2$ at convergence (harder task gets higher uncertainty → lower weight).

(d) Verify: the effective task weight $1/(2\sigma_k^2)$ is inversely proportional to $\mathcal{L}_k$.


In [ ]:
# Exercise 10: Your Solution

def multi_task_loss(L1, L2, log_sigma1, log_sigma2):
    """Kendall et al. 2018 multi-task loss. Inputs are scalars."""
    # YOUR CODE HERE
    pass

# (b) Zero-gradient check at optimal sigma
L1_val = 5.0
log_sigma1_opt = 0.5 * np.log(L1_val / 2)  # sigma^2 = L/2 -> log_sigma = 0.5*log(L/2)
eps_val = 1e-5
grad_num = None  # numerical gradient w.r.t log_sigma1 at optimum
print(f'Gradient at optimum: {grad_num} (should be ~0)')


In [ ]:
# Exercise 10: Solution

def multi_task_loss(L1, L2, log_sigma1, log_sigma2):
    s1 = np.exp(2 * log_sigma1)
    s2 = np.exp(2 * log_sigma2)
    return L1 / (2 * s1) + L2 / (2 * s2) + log_sigma1 + log_sigma2

header('Exercise 10: Multi-Task Uncertainty Weighting')

# (b) Zero gradient at optimum: d/d(log_sigma1) = -L1/sigma1^2 + 1 = 0 => sigma1^2 = L1
L1_val, L2_val = 5.0, 0.5
log_sigma1_opt = 0.5 * np.log(L1_val)  # sigma1^2 = L1 => log_sigma1 = 0.5*log(L1)
log_sigma2_opt = 0.5 * np.log(L2_val)
eps_val = 1e-5
grad1_num = (multi_task_loss(L1_val, L2_val, log_sigma1_opt + eps_val, log_sigma2_opt) -
             multi_task_loss(L1_val, L2_val, log_sigma1_opt - eps_val, log_sigma2_opt)) / (2*eps_val)
print(f'Gradient w.r.t. log_sigma1 at optimum: {grad1_num:.2e} (should be ~0)')
check_true('Gradient near zero at optimal sigma1', abs(grad1_num) < 1e-6)

# (c) Simulate optimisation over log_sigma
np.random.seed(42)
log_s1 = np.array([0.0])
log_s2 = np.array([0.0])
lr_s = 0.05

for step in range(500):
    s1 = np.exp(2 * log_s1[0])
    s2 = np.exp(2 * log_s2[0])
    # analytic gradients
    g1 = -L1_val / s1 + 1
    g2 = -L2_val / s2 + 1
    log_s1[0] -= lr_s * g1
    log_s2[0] -= lr_s * g2

sigma1_final = np.exp(log_s1[0])
sigma2_final = np.exp(log_s2[0])
w1_final = 1 / (2 * sigma1_final**2)
w2_final = 1 / (2 * sigma2_final**2)

print(f'\nL1={L1_val}, L2={L2_val}')
print(f'Optimal sigma1: {sigma1_final:.4f} (expected {np.sqrt(L1_val):.4f})')
print(f'Optimal sigma2: {sigma2_final:.4f} (expected {np.sqrt(L2_val):.4f})')
print(f'Effective weight w1: {w1_final:.4f}')
print(f'Effective weight w2: {w2_final:.4f}')
print(f'Weight ratio w2/w1: {w2_final/w1_final:.2f} (expected {L1_val/L2_val:.2f})')

check_close('sigma1 ≈ sqrt(L1)', sigma1_final, np.sqrt(L1_val), tol=0.01)
check_close('sigma2 ≈ sqrt(L2)', sigma2_final, np.sqrt(L2_val), tol=0.01)
check_true('sigma1 > sigma2 (harder task → higher uncertainty)', sigma1_final > sigma2_final)
check_true('w2 > w1 (easier task → higher effective weight)', w2_final > w1_final)

# (d) Inverse proportionality
ratio_weights = w2_final / w1_final
ratio_losses  = L1_val / L2_val
print(f'\nWeight ratio (w2/w1): {ratio_weights:.3f}')
print(f'Loss  ratio  (L1/L2): {ratio_losses:.3f}')
check_close('Weight ratio ≈ loss ratio (inverse prop)', ratio_weights, ratio_losses, tol=0.1)

print('\nTakeaway: Uncertainty weighting provides a principled alternative to hand-tuning '
      'task weights in multi-task learning. The log_sigma terms act as '
      'learned regularisers — larger task loss → larger sigma → smaller weight. '
      'Used in multi-modal LLMs (text + vision + audio) and NLP multi-task models.')


---

## What to Review After Finishing

- [ ] **Exercise 1–3**: Bayes-optimal statistics — mean (MSE), median (MAE), quantile. Can you derive each analytically by taking the derivative and setting it to zero?
- [ ] **Exercise 2**: Gaussian NLL = MSE when σ=1. Extend to multivariate Gaussian NLL (involves log det Σ).
- [ ] **Exercise 4**: ECE measures calibration mismatch. Know how temperature scaling (dividing logits by T) reduces ECE.
- [ ] **Exercise 5**: Forward KL is mode-covering (ELBO minimisation), reverse KL is mode-seeking (variational inference). Relate to Jensen's inequality.
- [ ] **Exercise 6**: Focal loss (Lin et al. 2017). Know how α (class weight) interacts with γ (focusing parameter).
- [ ] **Exercise 7**: InfoNCE lower-bounds mutual information. Temperature controls uniformity vs alignment trade-off.
- [ ] **Exercise 8**: DPO eliminates explicit reward model. Understand the connection to Bradley-Terry and RLHF.
- [ ] **Exercise 9**: Flat minima generalise better. SAM requires 2× forward passes — understand the trade-off.
- [ ] **Exercise 10**: Uncertainty weighting derives analytically. Extend to K tasks and classification likelihoods.

## References

1. Bartlett & Mendelson (2002). *Rademacher and Gaussian complexities.*
2. Gneiting & Raftery (2007). *Strictly proper scoring rules.*
3. Lin et al. (2017). *Focal loss for dense object detection.*
4. Oord et al. (2018). *Representation learning with CPC (InfoNCE).*
5. Kendall et al. (2018). *Multi-task learning using uncertainty.*
6. Foret et al. (2021). *Sharpness-Aware Minimisation (SAM).*
7. Rafailov et al. (2023). *Direct Preference Optimisation.*
8. Li et al. (2018). *Visualizing the loss landscape of neural nets.*
